# FeatureExtractor — Interactive Test Notebook

This notebook explores the `FeatureExtractor` (single-fn and multi-fn modes) and
`make_statistical_extractor` from the `tensorts` library.

**Sections:**
1. Single-fn mode
2. Multi-fn mode
3. `make_statistical_extractor`
4. Real-world pipeline simulations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from tensorts import (
    TimeTensor, Dim, FeatureExtractor, make_statistical_extractor,
    Pipeline, PreEmphasis, MelSpectrogram, LogTransform, StandardScaler,
    MinMaxScaler, RobustScaler, SlidingWindow, Flatten, STFT, ChannelSelect,
)

np.random.seed(0)
print("All imports successful.")

---
## Part 1: FeatureExtractor — Single-fn Mode

In single-fn mode, a single function is applied along `input_dim`.
The consumed axis is replaced by `output_dims`.

### 1a. Fake mel spectrogram (dense)

In [ ]:
def mel_like(x, n_mels=64, n_fft=512):
    """Fake mel-spectrogram: STFT power binned into n_mels bands."""
    stft = np.abs(np.fft.rfft(
        np.pad(x, (0, max(0, n_fft - len(x)))), n=n_fft
    )) ** 2
    frames = np.array_split(stft, n_mels)
    return np.array([f.sum() for f in frames])  # (n_mels,)

data = np.random.randn(4, 16000, 1)
ts = TimeTensor(data=data, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))

print(f"Input shape:  {ts.shape}")
print(f"Input dims:   {ts.dims}")

In [ ]:
fe = FeatureExtractor(
    fn=mel_like,
    input_dim=Dim.TIME,
    output_dims=(Dim.FREQUENCY,),
    fn_kwargs={"n_mels": 64},
    name="mel_like",
)
result = fe.fit_transform(ts)

print(f"Output shape: {result.shape}")
print(f"Output dims:  {result.dims}")
print(f"Has FREQUENCY dim: {result.has_dim(Dim.FREQUENCY)}")
print(f"Has TIME dim:      {result.has_dim(Dim.TIME)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.imshow(result.data[0, :, 0].reshape(-1, 1).T, aspect="auto", cmap="viridis")
ax.set_xlabel("Mel bin")
ax.set_title("Fake Mel Spectrogram — Sample 0")
plt.tight_layout()
plt.show()

### 1b. STFT stereo — both channels preserved

In [ ]:
data_stereo = np.random.randn(2, 8000, 2)
ts_stereo = TimeTensor(
    data=data_stereo,
    dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
    channels=["L", "R"],
)
print(f"Input shape: {ts_stereo.shape}, channels: {ts_stereo.channels.names}")

In [ ]:
stft_result = STFT(n_fft=512, hop_length=256).fit_transform(ts_stereo)

print(f"Output shape: {stft_result.shape}")
print(f"Output dims:  {stft_result.dims}")
print(f"n_channels:   {stft_result.n_channels}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ch_idx, ch_name in enumerate(["L", "R"]):
    spec = stft_result.data[0, :, :, ch_idx] if stft_result.data.ndim == 4 else stft_result.data[0, :, ch_idx]
    if spec.ndim == 1:
        axes[ch_idx].plot(spec)
        axes[ch_idx].set_title(f"STFT — {ch_name} channel")
    else:
        axes[ch_idx].imshow(spec, aspect="auto", origin="lower", cmap="magma")
        axes[ch_idx].set_title(f"STFT — {ch_name} channel")
        axes[ch_idx].set_ylabel("Frequency bin")
        axes[ch_idx].set_xlabel("Time frame")
plt.tight_layout()
plt.show()

### 1c. Resample via FeatureExtractor

`(T,) -> (T',)` — same dim type, different length.

In [ ]:
def resample_fn(x, target=64):
    indices = np.linspace(0, len(x) - 1, target)
    return np.interp(indices, np.arange(len(x)), x)

data_resamp = np.random.randn(5, 200, 3)
ts_resamp = TimeTensor(data=data_resamp, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))

fe_resamp = FeatureExtractor(
    fn=resample_fn, input_dim=Dim.TIME,
    output_dims=(Dim.TIME,), fn_kwargs={"target": 64},
)
result_resamp = fe_resamp.fit_transform(ts_resamp)

print(f"Before: {ts_resamp.shape}")
print(f"After:  {result_resamp.shape}")
print(f"Has TIME dim: {result_resamp.has_dim(Dim.TIME)}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(np.arange(200), ts_resamp.data[0, :, 0], label="Original (200 pts)", alpha=0.7)
ax.plot(np.linspace(0, 199, 64), result_resamp.data[0, :, 0], 'o-', label="Resampled (64 pts)", alpha=0.7, markersize=3)
ax.legend()
ax.set_title("Original vs Resampled Signal (Channel 0, Sample 0)")
ax.set_xlabel("Sample index")
plt.tight_layout()
plt.show()

### 1d. Scalar output

`output_dims=()`: `(B, T, C) -> (B, C)` — TIME axis removed entirely.

In [ ]:
energy_fe = FeatureExtractor(
    fn=lambda x: np.sum(x ** 2),
    input_dim=Dim.TIME,
    output_dims=(),
    name="energy",
)

data_scalar = np.random.randn(6, 100, 4)
ts_scalar = TimeTensor(data=data_scalar, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))
result_scalar = energy_fe.fit_transform(ts_scalar)

print(f"Before: {ts_scalar.shape}")
print(f"After:  {result_scalar.shape}")
print(f"Has TIME:    {result_scalar.has_dim(Dim.TIME)}")
print(f"Has CHANNEL: {result_scalar.has_dim(Dim.CHANNEL)}")
print(f"\nEnergy per sample per channel:")
print(result_scalar.data)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x_pos = np.arange(result_scalar.shape[0])
width = 0.18
for ch in range(result_scalar.shape[1]):
    ax.bar(x_pos + ch * width, result_scalar.data[:, ch], width, label=f"Channel {ch}")
ax.set_xlabel("Sample")
ax.set_ylabel("Energy")
ax.set_title("Energy per Channel per Sample")
ax.set_xticks(x_pos + width * 1.5)
ax.set_xticklabels([f"S{i}" for i in range(result_scalar.shape[0])])
ax.legend()
plt.tight_layout()
plt.show()

### 1e. Windowed input (B, W, T, C)

Leading dims are preserved: `(4, 10, 512, 1) -> (4, 10, 32, 1)`.

In [ ]:
def mel_stub(x, n_mels=32, n_fft=128):
    stft = np.abs(np.fft.rfft(np.pad(x, (0, max(0, n_fft - len(x)))), n=n_fft)) ** 2
    frames = np.array_split(stft, n_mels)
    return np.array([f.sum() for f in frames])

data_win = np.random.randn(4, 10, 512, 1)
ts_win = TimeTensor(data=data_win, dims=(Dim.BATCH, Dim.WINDOW, Dim.TIME, Dim.CHANNEL))

fe_win = FeatureExtractor(
    fn=mel_stub, input_dim=Dim.TIME,
    output_dims=(Dim.FREQUENCY,), fn_kwargs={"n_mels": 32},
)
result_win = fe_win.fit_transform(ts_win)

print(f"Input shape:  {ts_win.shape}")
print(f"Input dims:   {ts_win.dims}")
print(f"Output shape: {result_win.shape}")
print(f"Output dims:  {result_win.dims}")
print(f"Has WINDOW:    {result_win.has_dim(Dim.WINDOW)}")
print(f"Has FREQUENCY: {result_win.has_dim(Dim.FREQUENCY)}")

### 1f. Scalar on windowed: (B, W, T, C) -> (B, W, C)

RMS per window.

In [ ]:
data_ws = np.random.randn(4, 10, 128, 6)
ts_ws = TimeTensor(data=data_ws, dims=(Dim.BATCH, Dim.WINDOW, Dim.TIME, Dim.CHANNEL))

rms_fe = FeatureExtractor(
    fn=lambda x: float(np.sqrt(np.mean(x**2))),
    input_dim=Dim.TIME, output_dims=(),
)
result_ws = rms_fe.fit_transform(ts_ws)

print(f"Input shape:  {ts_ws.shape}")
print(f"Output shape: {result_ws.shape}")
print(f"Has TIME: {result_ws.has_dim(Dim.TIME)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(result_ws.data[0], aspect="auto", cmap="YlOrRd")
ax.set_xlabel("Channel")
ax.set_ylabel("Window")
ax.set_title("RMS per Window x Channel (Sample 0)")
plt.colorbar(im, ax=ax, label="RMS")
plt.tight_layout()
plt.show()

### 1g. Ragged single-fn: mel

Variable-length samples produce fixed-size output.

In [ ]:
def mel_stub_ragged(x, n_mels=32):
    stft = np.abs(np.fft.rfft(x, n=256)) ** 2
    return np.array([g.sum() for g in np.array_split(stft, n_mels)])

samples_ragged = [np.random.randn(np.random.randint(800, 2000)) for _ in range(5)]
ts_ragged = TimeTensor.from_ragged([s[:, np.newaxis] for s in samples_ragged])

print("Sample lengths before:")
for i, s in enumerate(samples_ragged):
    print(f"  Sample {i}: {len(s)} timesteps")
print(f"Is ragged: {ts_ragged.is_ragged}")

In [ ]:
fe_ragged = FeatureExtractor(
    fn=mel_stub_ragged, input_dim=Dim.TIME,
    output_dims=(Dim.FREQUENCY,),
)
result_ragged = fe_ragged.fit_transform(ts_ragged)

print(f"Output shape: {result_ragged.shape}")
print(f"Is ragged:    {result_ragged.is_ragged}")
print("All samples now have uniform shape (32 mel bins).")

### 1h. Ragged scalar: energy per channel

`y` labels are preserved through the transform.

In [ ]:
samples_rag_sc = [np.random.randn(np.random.randint(50, 200), 3) for _ in range(6)]
y_rag = np.array([0, 1, 2, 0, 1, 2])
ts_rag_sc = TimeTensor.from_ragged(samples_rag_sc, y=y_rag)

fe_rag_sc = FeatureExtractor(
    fn=lambda x: np.sum(x**2), input_dim=Dim.TIME, output_dims=(),
)
result_rag_sc = fe_rag_sc.fit_transform(ts_rag_sc)

print(f"Output shape: {result_rag_sc.shape}")
print(f"y preserved:  {np.array_equal(result_rag_sc.y, y_rag)}")
print(f"y values:     {result_rag_sc.y}")
print(f"\nEnergy per sample per channel:")
print(result_rag_sc.data)

---
## Part 2: FeatureExtractor — Multi-fn Mode

In multi-fn mode, each function returns a scalar or 1-D vector per channel.
Results are concatenated along `Dim.FEATURE`.

### 2a. All-scalar functions (dense)

In [ ]:
data_mf = np.random.randn(6, 100, 3)
ts_mf = TimeTensor(
    data=data_mf, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
    channels=["a", "b", "c"],
)

fe_mf = FeatureExtractor(
    functions=[
        ("mean",   np.mean),
        ("std",    np.std),
        ("energy", lambda x: float(np.sum(x**2))),
    ],
    input_dim=Dim.TIME,
)
result_mf = fe_mf.fit_transform(ts_mf)

print(f"Output shape:       {result_mf.shape}  (B=6, K=3 features, C=3 channels)")
print(f"Output dims:        {result_mf.dims}")
print(f"feature_names_out_: {fe_mf.feature_names_out_}")
print(f"Has FEATURE dim:    {result_mf.has_dim(Dim.FEATURE)}")
print(f"Has TIME dim:       {result_mf.has_dim(Dim.TIME)}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
feature_names = fe_mf.feature_names_out_
n_features = len(feature_names)
n_channels = result_mf.shape[2]
x_pos = np.arange(n_features)
width = 0.25

for s in range(min(3, result_mf.shape[0])):
    for ch in range(n_channels):
        offset = (s * n_channels + ch) * width / n_channels
        ax.bar(x_pos + offset, result_mf.data[s, :, ch], width / n_channels,
               label=f"S{s}/Ch{ch}" if s < 2 else None, alpha=0.7)

ax.set_xticks(x_pos + width / 2)
ax.set_xticklabels(feature_names)
ax.set_title("Multi-fn All-Scalar: 3 Features x 3 Channels (first 2 samples)")
ax.set_ylabel("Value")
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

### 2b. Mixed scalar + vector function

In [ ]:
def top3_fft(x):
    return np.sort(np.abs(np.fft.rfft(x)))[-3:]

data_mv = np.random.randn(4, 128, 2)
ts_mv = TimeTensor(data=data_mv, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))

fe_mv = FeatureExtractor(
    functions=[
        ("mean", np.mean),       # scalar -> K=1
        ("top3", top3_fft),      # vector -> K=3
    ],
    input_dim=Dim.TIME,
)
result_mv = fe_mv.fit_transform(ts_mv)

print(f"Output shape:       {result_mv.shape}  (B=4, K_total=1+3=4, C=2)")
print(f"feature_names_out_: {fe_mv.feature_names_out_}")

### 2c. Windowed multi-fn: (B, W, T, C) -> (B, W, K, C)

In [ ]:
data_wmf = np.random.randn(3, 8, 64, 4)
ts_wmf = TimeTensor(data=data_wmf, dims=(Dim.BATCH, Dim.WINDOW, Dim.TIME, Dim.CHANNEL))

fe_wmf = FeatureExtractor(
    functions=[
        ("mean", np.mean),
        ("rms", lambda x: float(np.sqrt(np.mean(x**2)))),
    ],
    input_dim=Dim.TIME,
)
result_wmf = fe_wmf.fit_transform(ts_wmf)

print(f"Input:  {ts_wmf.shape}  dims={ts_wmf.dims}")
print(f"Output: {result_wmf.shape}  dims={result_wmf.dims}")
print(f"Has WINDOW:  {result_wmf.has_dim(Dim.WINDOW)}")
print(f"Has FEATURE: {result_wmf.has_dim(Dim.FEATURE)}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for feat_idx, feat_name in enumerate(fe_wmf.feature_names_out_):
    for ch in range(min(2, result_wmf.shape[3])):
        ax.plot(result_wmf.data[0, :, feat_idx, ch],
                label=f"{feat_name} / Ch{ch}", marker='o', markersize=4)
ax.set_xlabel("Window")
ax.set_ylabel("Feature value")
ax.set_title("Feature Values Across Windows (Sample 0)")
ax.legend()
plt.tight_layout()
plt.show()

### 2d. Ragged multi-fn

In [ ]:
samples_rmf = [np.random.randn(100 + i * 50, 3) for i in range(5)]
y_rmf = np.array([0, 1, 0, 1, 0])
ts_rmf = TimeTensor.from_ragged(samples_rmf, channels=["x", "y", "z"], y=y_rmf)

fe_rmf = FeatureExtractor(
    functions=[("mean", np.mean), ("std", np.std), ("min", np.min)],
    input_dim=Dim.TIME,
)
result_rmf = fe_rmf.fit_transform(ts_rmf)

print(f"Output shape: {result_rmf.shape}  (B=5, K=3 fns, C=3 channels)")
print(f"Is ragged:    {result_rmf.is_ragged}")
print(f"y preserved:  {np.array_equal(result_rmf.y, y_rmf)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for feat_idx, feat_name in enumerate(fe_rmf.feature_names_out_):
    bp_data = [result_rmf.data[:, feat_idx, ch] for ch in range(3)]
    axes[feat_idx].boxplot(bp_data, labels=["x", "y", "z"])
    axes[feat_idx].set_title(f"{feat_name} across samples")
    axes[feat_idx].set_ylabel("Value")
plt.suptitle("Ragged Multi-fn: Feature Distributions per Channel")
plt.tight_layout()
plt.show()

### 2e. Per-function kwargs (3-tuples)

In [ ]:
def top_k(x, k=5):
    return np.sort(np.abs(x))[-k:]

data_kw = np.random.randn(3, 100, 2)
ts_kw = TimeTensor(data=data_kw, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))

fe_kw = FeatureExtractor(
    functions=[
        ("mean",  np.mean),
        ("top3",  top_k, {"k": 3}),
        ("top5",  top_k, {"k": 5}),
    ],
    input_dim=Dim.TIME,
)
result_kw = fe_kw.fit_transform(ts_kw)

print(f"Output shape: {result_kw.shape}  (K_total = 1 + 3 + 5 = 9)")
print(f"\nAll feature names:")
for name in fe_kw.feature_names_out_:
    print(f"  {name}")

### 2f. Channel names preserved

In [ ]:
data_ch = np.random.randn(4, 100, 3)
ts_ch = TimeTensor(
    data=data_ch, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
    channels=["acc_x", "acc_y", "acc_z"],
)

fe_ch = FeatureExtractor(
    functions=[("mean", np.mean), ("std", np.std)],
    input_dim=Dim.TIME,
)
result_ch = fe_ch.fit_transform(ts_ch)

print(f"Output shape:   {result_ch.shape}")
print(f"Channel names:  {result_ch.channels.names}")
print(f"Feature count:  {result_ch.shape[1]}")

### 2g. get_params / set_params / clone

In [ ]:
from sklearn.base import clone

fe_clone = FeatureExtractor(
    functions=[("mean", np.mean), ("std", np.std)],
    input_dim=Dim.TIME,
)

params = fe_clone.get_params()
print("get_params():")
for k, v in params.items():
    print(f"  {k}: {v}")

fe_cloned = clone(fe_clone)
print(f"\nCloned successfully: {fe_cloned.functions is not None}")
print(f"Clone mode: {fe_cloned._mode}")

### 2h. Error cases

In [ ]:
# Error 1: fn AND functions both set -> ValueError
try:
    FeatureExtractor(fn=np.mean, functions=[("mean", np.mean)], input_dim=Dim.TIME)
    print("ERROR: Should have raised ValueError!")
except ValueError as e:
    print(f"Correctly raised ValueError: {e}")

In [ ]:
# Error 2: functions=[] -> ValueError
try:
    FeatureExtractor(functions=[], input_dim=Dim.TIME)
    print("ERROR: Should have raised ValueError!")
except ValueError as e:
    print(f"Correctly raised ValueError: {e}")

---
## Part 3: make_statistical_extractor

Factory function that creates a multi-fn FeatureExtractor pre-loaded with
built-in statistical functions and/or custom callables.

### 3a. Default features

In [ ]:
data_stat = np.random.randn(5, 100, 3)
ts_stat = TimeTensor(data=data_stat, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))

fe_stat = make_statistical_extractor()
result_stat = fe_stat.fit_transform(ts_stat)

print(f"Output shape:       {result_stat.shape}  (B=5, K=4 defaults, C=3)")
print(f"feature_names_out_: {fe_stat.feature_names_out_}")

### 3b. All 13 built-in features

In [ ]:
all_features = [
    "mean", "std", "var", "min", "max", "median",
    "rms", "range", "energy", "skew", "kurtosis", "iqr", "zcr",
]

data_all = np.random.randn(4, 200, 2)
ts_all = TimeTensor(data=data_all, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))

fe_all = make_statistical_extractor(all_features)
result_all = fe_all.fit_transform(ts_all)

print(f"Output shape:       {result_all.shape}  (B=4, K=13, C=2)")
print(f"feature_names_out_: {fe_all.feature_names_out_}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
im = ax.imshow(result_all.data[0, :, :], aspect="auto", cmap="coolwarm")
ax.set_yticks(range(13))
ax.set_yticklabels(all_features)
ax.set_xticks(range(2))
ax.set_xticklabels([f"Ch {i}" for i in range(2)])
ax.set_title("All 13 Statistical Features (Sample 0)")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(18, 9))
axes = axes.flatten()
for feat_idx in range(13):
    vals = result_all.data[:, feat_idx, :]  # (B, C)
    axes[feat_idx].boxplot(vals)
    axes[feat_idx].set_title(all_features[feat_idx], fontsize=10)
    axes[feat_idx].set_xticklabels([f"Ch{i}" for i in range(vals.shape[1])], fontsize=8)
# Hide unused subplots
for idx in range(13, 15):
    axes[idx].set_visible(False)
plt.suptitle("Distribution of Each Feature Across Samples", fontsize=13)
plt.tight_layout()
plt.show()

### 3c. Ragged input

In [ ]:
samples_stat_rag = [np.random.randn(np.random.randint(80, 200), 4) for _ in range(10)]
y_stat_rag = np.random.randint(0, 2, 10)
ts_stat_rag = TimeTensor.from_ragged(samples_stat_rag, y=y_stat_rag)

fe_stat_rag = make_statistical_extractor(["mean", "std", "rms"])
result_stat_rag = fe_stat_rag.fit_transform(ts_stat_rag)

print(f"Output shape: {result_stat_rag.shape}  (B=10, K=3, C=4)")
print(f"y preserved:  {np.array_equal(result_stat_rag.y, y_stat_rag)}")
print(f"y values:     {result_stat_rag.y}")

### 3d. Custom callable mixed with built-in

In [ ]:
def dominant_freq(x):
    freqs = np.fft.rfftfreq(len(x))
    mag = np.abs(np.fft.rfft(x))
    return float(freqs[np.argmax(mag)])

data_custom = np.random.randn(4, 128, 2)
ts_custom = TimeTensor(data=data_custom, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))

fe_custom = make_statistical_extractor([
    "mean",
    ("dominant_freq", dominant_freq),
])
result_custom = fe_custom.fit_transform(ts_custom)

print(f"Output shape:       {result_custom.shape}  (B=4, K=2, C=2)")
print(f"feature_names_out_: {fe_custom.feature_names_out_}")
print(f"\nValues (sample 0):")
for i, name in enumerate(fe_custom.feature_names_out_):
    print(f"  {name}: {result_custom.data[0, i, :]}")

### 3e. Custom with kwargs (3-tuple)

In [ ]:
def top_k_energy(x, k=3):
    return np.sort(np.abs(np.fft.rfft(x)))[-k:]

data_3t = np.random.randn(3, 128, 2)
ts_3t = TimeTensor(data=data_3t, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL))

fe_3t = make_statistical_extractor([
    "mean",
    ("top5_fft", top_k_energy, {"k": 5}),
])
result_3t = fe_3t.fit_transform(ts_3t)

print(f"Output shape: {result_3t.shape}  (K = 1 + 5 = 6)")
print(f"\nFeature names:")
for name in fe_3t.feature_names_out_:
    print(f"  {name}")

### 3f. Error cases

In [ ]:
# Unknown string -> ValueError
try:
    make_statistical_extractor(["mean", "invalid_feature"])
    print("ERROR: Should have raised ValueError!")
except ValueError as e:
    print(f"Correctly raised ValueError: {e}")

In [ ]:
# Bare function without name -> TypeError
try:
    make_statistical_extractor([np.mean])
    print("ERROR: Should have raised TypeError!")
except TypeError as e:
    print(f"Correctly raised TypeError: {e}")

---
## Part 4: Real-world Pipeline Simulations

Each step is executed individually with prints and plots to trace the
data flow through the pipeline.

### 4a. Audio -> Mel Pipeline (step by step)

In [ ]:
# Create data
sr = 16000
data_audio = np.random.randn(20, sr, 1)
y_audio = np.random.randint(0, 2, 20)
ts_audio = TimeTensor(
    data=data_audio, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
    channels=["audio"], y=y_audio, metadata={"sampling_rate": sr},
)
print(f"Original: {ts_audio.shape}, dims={ts_audio.dims}")

In [ ]:
# Step 1: PreEmphasis
X1 = PreEmphasis(coeff=0.97).fit_transform(ts_audio)
print(f"After PreEmphasis: {X1.shape}")

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(ts_audio.data[0, :500, 0], label="Original", alpha=0.7)
ax.plot(X1.data[0, :500, 0], label="Pre-emphasized", alpha=0.7)
ax.legend()
ax.set_title("PreEmphasis: Original vs Pre-emphasized (first 500 samples)")
ax.set_xlabel("Sample")
plt.tight_layout()
plt.show()

In [ ]:
# Step 2: MelSpectrogram
X2 = MelSpectrogram(n_fft=512, hop_length=160, n_mels=64, sampling_rate=sr).fit_transform(X1)
print(f"After MelSpectrogram: {X2.shape}, dims={X2.dims}")

fig, ax = plt.subplots(figsize=(10, 4))
mel_data = X2.data[0, :, :, 0] if X2.data.ndim == 4 else X2.data[0, :, 0]
if mel_data.ndim == 2:
    ax.imshow(mel_data, aspect="auto", origin="lower", cmap="viridis")
    ax.set_ylabel("Mel bin")
    ax.set_xlabel("Time frame")
else:
    ax.plot(mel_data)
    ax.set_xlabel("Mel bin")
ax.set_title("Mel Spectrogram (Sample 0)")
plt.tight_layout()
plt.show()

In [ ]:
# Step 3: LogTransform
X3 = LogTransform().fit_transform(X2)
print(f"After LogTransform: {X3.shape}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax_idx, (label, Xdata) in enumerate([("Before Log", X2), ("After Log", X3)]):
    d = Xdata.data[0, :, :, 0] if Xdata.data.ndim == 4 else Xdata.data[0, :, 0]
    if d.ndim == 2:
        axes[ax_idx].imshow(d, aspect="auto", origin="lower", cmap="viridis")
        axes[ax_idx].set_ylabel("Mel bin")
        axes[ax_idx].set_xlabel("Time frame")
    else:
        axes[ax_idx].plot(d)
    axes[ax_idx].set_title(label)
plt.tight_layout()
plt.show()

In [ ]:
# Step 4: StandardScaler
X4 = StandardScaler().fit_transform(X3)
print(f"After StandardScaler: {X4.shape}, mean={X4.data.mean():.4f}, std={X4.data.std():.4f}")
print(f"y preserved: {np.array_equal(X4.y, y_audio)}")

### 4b. HAR Ragged -> Stats Pipeline (step by step)

In [ ]:
# Create ragged HAR data
rng = np.random.default_rng(1)
har_samples = [rng.standard_normal((rng.integers(150, 400), 6)) for _ in range(40)]
har_labels = np.tile(np.arange(4), 10)

ts_har = TimeTensor.from_ragged(
    har_samples,
    channels=["ax", "ay", "az", "gx", "gy", "gz"],
    y=har_labels,
)
print(f"Original: {ts_har.n_samples} ragged samples, is_ragged={ts_har.is_ragged}")
print(f"y: {ts_har.y[:10]}...")

In [ ]:
# Step 1: StandardScaler on ragged
har_X1 = StandardScaler().fit_transform(ts_har)
print(f"After StandardScaler: n_samples={har_X1.n_samples}, is_ragged={har_X1.is_ragged}")
print(f"y preserved: {np.array_equal(har_X1.y, har_labels)}")

In [ ]:
# Step 2: SlidingWindow (ragged -> dense)
har_X2 = SlidingWindow(size=64, stride=32, label_strategy="repeat").fit_transform(har_X1)
print(f"After SlidingWindow: shape={har_X2.shape}, dims={har_X2.dims}")
print(f"y shape: {har_X2.y.shape if har_X2.y is not None else None}")

In [ ]:
# Step 3: make_statistical_extractor
har_fe = make_statistical_extractor(["mean", "std", "rms", "zcr", "energy"])
har_X3 = har_fe.fit_transform(har_X2)
print(f"After stats extractor: shape={har_X3.shape}")
print(f"feature_names_out_: {har_fe.feature_names_out_}")
print(f"y preserved: {har_X3.y is not None}")

In [ ]:
# Step 4: Flatten
har_X4 = Flatten().fit_transform(har_X3)
print(f"After Flatten: shape={har_X4.shape}")
print(f"Expected last dim: 5 stats x 6 channels = 30")
print(f"y preserved: {har_X4.y is not None}")

### 4c. EEG with ChannelSelect (step by step)

In [ ]:
# Create EEG data
eeg_sr = 256
data_eeg = np.random.randn(10, eeg_sr * 2, 8)
ts_eeg = TimeTensor(
    data=data_eeg, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
    channels=["Fp1", "Fp2", "C3", "C4", "P3", "P4", "O1", "O2"],
    channel_groups={"motor": ["C3", "C4"]},
)
print(f"EEG input: {ts_eeg.shape}")
print(f"Channels: {ts_eeg.channels.names}")

In [ ]:
# Step 1: STFT
eeg_X1 = STFT(n_fft=64, hop_length=32).fit_transform(ts_eeg)
print(f"After STFT: {eeg_X1.shape}, dims={eeg_X1.dims}")

In [ ]:
# Step 2: ChannelSelect (motor cortex) + StandardScaler
eeg_X2 = ChannelSelect("motor", StandardScaler()).fit_transform(eeg_X1)
print(f"After ChannelSelect(motor) + Scale: {eeg_X2.shape}")
print(f"n_channels: {eeg_X2.n_channels} (all kept, motor scaled)")

### 4d. Multi-fn in windowed ragged pipeline (step by step)

In [ ]:
# Create ragged sensor data
def autocorr_lag1(x):
    if len(x) < 2:
        return np.float64(0.0)
    return np.float64(np.corrcoef(x[:-1], x[1:])[0, 1])

rng_d = np.random.default_rng(42)
sensor_samples = [rng_d.standard_normal((rng_d.integers(200, 500), 4)) for _ in range(20)]
y_sensor = np.random.randint(0, 3, 20)
ts_sensor = TimeTensor.from_ragged(
    sensor_samples, y=y_sensor, channels=["a", "b", "c", "d"],
)
print(f"Ragged sensor data: {ts_sensor.n_samples} samples, is_ragged={ts_sensor.is_ragged}")

In [ ]:
# Step 1: RobustScaler
sensor_X1 = RobustScaler().fit_transform(ts_sensor)
print(f"After RobustScaler: n_samples={sensor_X1.n_samples}, is_ragged={sensor_X1.is_ragged}")

In [ ]:
# Step 2: SlidingWindow
sensor_X2 = SlidingWindow(size=50, stride=25, label_strategy="repeat").fit_transform(sensor_X1)
print(f"After SlidingWindow: shape={sensor_X2.shape}, dims={sensor_X2.dims}")

In [ ]:
# Step 3: Multi-fn FeatureExtractor
fe_sensor = FeatureExtractor(
    functions=[
        ("mean",  np.mean),
        ("std",   np.std),
        ("lag1",  autocorr_lag1),
    ],
    input_dim=Dim.TIME,
)
sensor_X3 = fe_sensor.fit_transform(sensor_X2)
print(f"After FeatureExtractor: shape={sensor_X3.shape}")
print(f"feature_names_out_: {fe_sensor.feature_names_out_}")

In [ ]:
# Step 4: Flatten
sensor_X4 = Flatten().fit_transform(sensor_X3)
print(f"After Flatten: shape={sensor_X4.shape}")
print(f"Expected last dim: 3 features x 4 channels = 12")
print(f"y preserved: {sensor_X4.y is not None}")
print(f"\nFull pipeline: {ts_sensor.n_samples} ragged samples -> {sensor_X4.shape}")

### 4e. make_statistical_extractor with mixed built-ins + custom in pipeline

In [ ]:
def fft_energy(x):
    return float(np.sum(np.abs(np.fft.rfft(x)) ** 2))

data_pipe = np.random.randn(10, 200, 3)
y_pipe = np.random.randint(0, 2, 10)
ts_pipe = TimeTensor(
    data=data_pipe, dims=(Dim.BATCH, Dim.TIME, Dim.CHANNEL),
    channels=["x", "y", "z"], y=y_pipe,
)
print(f"Input: {ts_pipe.shape}")

In [ ]:
fe_mixed = make_statistical_extractor([
    "mean", "std",
    ("fft_energy", fft_energy),
])

pipe_full = Pipeline([
    ("scale",  StandardScaler()),
    ("window", SlidingWindow(size=50, stride=25)),
    ("fe",     fe_mixed),
    ("flat",   Flatten()),
])
result_pipe = pipe_full.fit_transform(ts_pipe)

print(f"Output shape: {result_pipe.shape}")
print(f"Last dim divisible by 9 (3 fns x 3 ch): {result_pipe.shape[-1] % 9 == 0}")
print(f"feature_names_out_: {fe_mixed.feature_names_out_}")

---
## Summary

All sections of the FeatureExtractor API have been explored:

1. **Single-fn mode**: mel spectrogram, STFT stereo, resample, scalar, windowed, ragged
2. **Multi-fn mode**: all-scalar, mixed scalar+vector, windowed, ragged, kwargs, channels, clone, errors
3. **make_statistical_extractor**: defaults, all 13 built-ins, ragged, custom callable, kwargs, errors
4. **Pipelines**: Audio-Mel, HAR ragged-stats, EEG channel-select, multi-fn windowed ragged